# Beijing air-quality statistical analysis

This notebook summarizes daily pollutant levels across Beijing monitoring stations, reports descriptive statistics and confidence intervals, and explores station differences with ANOVA and Tukey HSD. The results are exploratory and depend on the current `air_quality.aqi_daily` data; they do not establish causes.

## 1. Import libraries

Load the libraries used for data processing, statistical tests, database access, and confidence intervals.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sqlalchemy import create_engine
import glob
import os
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

## 2. Load daily records

Read station-day pollutant and weather summaries from MySQL `air_quality.aqi_daily`. The descriptive statistics and station tests below use the records in this table, so results depend on its current date range, station coverage, and missing values.

In [ ]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")

    df = pd.read_sql_query("SELECT * FROM aqi_daily", engine)
    combined_df = df.copy()
    print(f"Loaded {len(df):,} daily station rows from aqi_daily.")
except Exception as error:
    raise RuntimeError(
        "Could not connect to MySQL or load the aqi_daily table. "
        "Check that MySQL is running and air_quality.aqi_daily exists."
    ) from error

## 3. Descriptive statistics

For each pollutant, calculate the non-null observation count, mean, median, standard deviation, variance, minimum, maximum, first and third quartiles, and interquartile range (IQR = Q3 − Q1). The next output gives a 95% t-based confidence interval for each overall mean. These summaries treat the available station-day values as the observed sample; repeated dates and differences in station coverage should be considered when interpreting uncertainty.

In [ ]:
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

descriptive_stats = df[pollutants].agg([
    "count",
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max"
])

q1 = df[pollutants].quantile(0.25)
q3 = df[pollutants].quantile(0.75)

descriptive_stats.loc["Q1"] = q1
descriptive_stats.loc["Q3"] = q3
descriptive_stats.loc["IQR"] = q3 - q1

descriptive_stats = descriptive_stats.T

descriptive_stats

In [ ]:
confidence_results = []

for pollutant in pollutants:
    values = df[pollutant].dropna()
    n = len(values)
    mean = values.mean()
    standard_error = values.std() / np.sqrt(n)
    ci_low, ci_high = sm.stats.DescrStatsW(values).tconfint_mean(
        alpha=0.05
    )

    confidence_results.append({
        "Pollutant": pollutant,
        "Mean": mean,
        "95% CI Lower": ci_low,
        "95% CI Upper": ci_high
    })

confidence_df = pd.DataFrame(confidence_results)

confidence_df

## 4. Station-level summaries

Summarize each pollutant by station using its mean, median, and standard deviation. The following tables identify each pollutant's highest station mean and report t-based 95% confidence intervals where a station has at least two valid observations. These are descriptive comparisons; the intervals do not adjust for testing multiple stations or account for shared conditions on the same dates.

In [ ]:
station_stats = df.groupby("station")[pollutants].agg([
    "mean",
    "median",
    "std"
])

station_stats

In [ ]:
station_summary = []

for station, group in df.groupby("station"):
    for pollutant in pollutants:
        values = group[pollutant].dropna()
        station_summary.append({
            "Station": station,
            "Pollutant": pollutant,
            "Mean": values.mean(),
            "Median": values.median(),
            "Standard Deviation": values.std()
        })

station_summary_df = pd.DataFrame(station_summary)

station_summary_df

In [ ]:
highest_mean_results = []

for pollutant in pollutants:
    station_means = df.groupby("station")[pollutant].mean()
    highest_station = station_means.idxmax()
    highest_mean = station_means.max()

    highest_mean_results.append({
        "Pollutant": pollutant,
        "Station with Highest Mean": highest_station,
        "Mean Concentration": highest_mean
    })

highest_mean_df = pd.DataFrame(highest_mean_results)

highest_mean_df

In [ ]:
station_confidence = []

for station, group in df.groupby("station"):
    for pollutant in pollutants:
        values = group[pollutant].dropna()

        if len(values) > 1:
            ci_low, ci_high = sm.stats.DescrStatsW(
                values
            ).tconfint_mean(alpha=0.05)

            station_confidence.append({
                "Station": station,
                "Pollutant": pollutant,
                "Mean": values.mean(),
                "95% CI Lower": ci_low,
                "95% CI Upper": ci_high
            })

station_confidence_df = pd.DataFrame(station_confidence)

pd.set_option("display.max_rows", None)

## 5. Station differences: one-way ANOVA

For each pollutant, one-way ANOVA tests whether mean daily summaries differ among stations. Bonferroni adjustment is applied across the six pollutant-level ANOVA p-values. The next section runs Tukey HSD station-pair comparisons for each pollutant and reports Levene and residual normality diagnostics. These tests assume independent observations and, for standard ANOVA/Tukey inference, comparable within-group variances and approximately normal residuals. Because stations share regional conditions on the same dates and records are repeated over time, treat p-values as exploratory.

In [ ]:
from scipy.stats import f_oneway, levene, normaltest
from statsmodels.stats.multicomp import pairwise_tukeyhsd

pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

for pollutant in pollutants:
    df[pollutant] = pd.to_numeric(df[pollutant], errors="coerce")

anova_results = []

for pollutant in pollutants:
    station_groups = [
        group[pollutant].dropna().to_numpy()
        for _, group in df.groupby("station", sort=True)
    ]
    f_statistic, p_value = f_oneway(*station_groups)
    anova_results.append({
        "pollutant": pollutant,
        "stations": len(station_groups),
        "observations": sum(len(values) for values in station_groups),
        "F-statistic": f_statistic,
        "p-value": p_value,
    })

anova_results = pd.DataFrame(anova_results)
anova_results["p-value (Bonferroni)"] = multipletests(
    anova_results["p-value"], method="bonferroni"
)[1]
anova_results["significant (alpha=0.05)"] = (
    anova_results["p-value (Bonferroni)"] < 0.05
)

anova_results.round({
    "F-statistic": 3,
    "p-value": 6,
    "p-value (Bonferroni)": 6,
})

## 6. ANOVA interpretation

Use the Bonferroni-adjusted p-values to assess evidence of differences in mean daily summaries across stations for each pollutant. A small p-value indicates that not all station means are equal under the test assumptions; it does not identify which stations differ. Shared date-level conditions and serial dependence may make the nominal p-values too optimistic.

## 7. Pairwise comparisons and assumption checks

Tukey HSD is run for every pollutant and compares all station pairs while controlling the family-wise error rate within that pollutant's pairwise comparisons. The output retains only pairs marked significant by Tukey's procedure. Levene's test checks for differences in station variances. A normality test is applied to a random sample of up to 5,000 residuals after subtracting each station's mean. These diagnostics assess assumptions but do not correct the tests for dependence between stations or dates.

## 8. Compact post-hoc results

Review the count of significant Tukey pairs for each pollutant, the three largest significant absolute mean differences per pollutant, and the assumption-test results. Mean differences are reported as `group2 − group1` by the Tukey output; use the confidence bounds and adjusted p-values alongside their magnitude.

In [ ]:
significant_pairwise_results = []
assumption_results = []

for pollutant in pollutants:
    valid_data = df[["station", pollutant]].dropna()

    tukey = pairwise_tukeyhsd(
        endog=valid_data[pollutant],
        groups=valid_data["station"],
        alpha=0.05,
    )
    tukey_table = pd.DataFrame(
        tukey.summary().data[1:],
        columns=tukey.summary().data[0],
    )
    tukey_table = tukey_table[tukey_table["reject"] == True].copy()
    tukey_table.insert(0, "pollutant", pollutant)
    significant_pairwise_results.append(tukey_table)

    station_values = [
        group[pollutant].dropna().to_numpy()
        for _, group in df.groupby("station", sort=True)
    ]
    levene_statistic, levene_p_value = levene(*station_values, center="median")

    residuals = valid_data[pollutant] - valid_data.groupby("station")[pollutant].transform("mean")
    residual_sample = residuals.sample(
        n=min(5000, len(residuals)), random_state=42
    )
    normality_statistic, normality_p_value = normaltest(residual_sample)

    assumption_results.append({
        "pollutant": pollutant,
        "Levene statistic": levene_statistic,
        "Levene p-value": levene_p_value,
        "Residual normality statistic": normality_statistic,
        "Residual normality p-value": normality_p_value,
        "equal variances at alpha=0.05": levene_p_value >= 0.05,
        "normal residuals at alpha=0.05": normality_p_value >= 0.05,
    })

significant_pairwise_results = pd.concat(
    significant_pairwise_results, ignore_index=True
)
assumption_results = pd.DataFrame(assumption_results)

pairwise_counts = (
    significant_pairwise_results.groupby("pollutant")
    .size()
    .reset_index(name="significant pair count")
)

top_pairwise_differences = (
    significant_pairwise_results.assign(
        absolute_meandiff=lambda table: table["meandiff"].abs()
    )
    .sort_values(["pollutant", "absolute_meandiff"], ascending=[True, False])
    .groupby("pollutant", group_keys=False)
    .head(3)
    [["pollutant", "group1", "group2", "meandiff", "p-adj", "lower", "upper"]]
)

print("Number of significant Tukey HSD pairs by pollutant")
display(pairwise_counts)

print("Three largest significant mean differences per pollutant")
display(top_pairwise_differences)

print("Assumption test results")
display(assumption_results.round(6))

## 9. Post-hoc interpretation

Use the current Tukey HSD output to see which station pairs have statistically distinguishable daily pollutant means under the test assumptions. Consider the size and direction of each mean difference, its confidence interval, and the diagnostic results. Findings depend on the current `aqi_daily` records and should be treated as exploratory because the tests do not model shared date effects or time dependence.